In [3]:
# load data
import pandas as pd

df = pd.read_csv('instagram_global_influencers_2026.csv')
df

,rank,channel_info,influence_score,posts,followers,avg_likes,60_day_eng_rate,new_post_avg_likes,total_likes,country,category
0,1,cristiano,98,3782,680000000,12800000,1.89,14200000,48400000000,Portugal,Sports
1,2,leomessi,97,1204,517000000,14300000,2.77,16100000,17200000000,Argentina,Sports
2,3,selenagomez,96,1935,403000000,8700000,2.16,9400000,16800000000,United States,Music/Entertainment
3,4,therock,95,7842,381000000,4200000,1.10,4800000,32900000000,United States,Entertainment/Fitness
4,5,kyliejenner,94,7216,380000000,7500000,1.97,8100000,54100000000,United States,Fashion/Business
...,...,...,...,...,...,...,...,...,...,...,...
95,96,sachintendulkar,34,3824,43800000,620000,1.42,680000,2370000000,India,Sports/Cricket
96,97,kaikigreene,33,4216,8400000,480000,5.71,530000,2020000000,United States,Fitness/Wellness
97,98,bellapoarch,32,624,33400000,2800000,8.38,3100000,1750000000,Philippines/United States,Content Creator
98,99,soheeart,31,2418,18600000,520000,2.80,580000,1260000000,South Korea,Art/Design


In [4]:
# EDA
print(f'Cek 5 data teratas \n {df.head()} \n')

print(f'Cek tipe data \n {df.dtypes} \n')

print(f'Cek missing values \n {df.isnull().sum()} \n')\

print(f'Cek duplicate data \n {df.duplicated().sum()}')

Cek 5 data teratas 
    rank channel_info  influence_score  posts  followers  avg_likes  \
0     1    cristiano               98   3782  680000000   12800000   
1     2     leomessi               97   1204  517000000   14300000   
2     3  selenagomez               96   1935  403000000    8700000   
3     4      therock               95   7842  381000000    4200000   
4     5  kyliejenner               94   7216  380000000    7500000   

   60_day_eng_rate  new_post_avg_likes  total_likes        country  \
0             1.89            14200000  48400000000       Portugal   
1             2.77            16100000  17200000000      Argentina   
2             2.16             9400000  16800000000  United States   
3             1.10             4800000  32900000000  United States   
4             1.97             8100000  54100000000  United States   

                category  
0                 Sports  
1                 Sports  
2    Music/Entertainment  
3  Entertainment/Fitness  
4 

In [5]:
# data cleaning
df = df.drop(columns='rank')

In [7]:
# outlier
numerical_columns = df.select_dtypes(include='number').columns

outlier_summary = []

for column in numerical_columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]

    outlier_summary.append({
        'kolom': column,
        'Q1': Q1,
        'Q3': Q3,
        'IQR': IQR,
        'batas_bawah': lower_bound,
        'batas_atas': upper_bound,
        'jumlah_outlier': len(outliers),
        'persen_outlier': round(len(outliers) / len(df) * 100, 2)
    })

outlier_df = pd.DataFrame(outlier_summary)
outlier_df

,kolom,Q1,Q3,IQR,batas_bawah,batas_atas,jumlah_outlier,persen_outlier
0,influence_score,5.000000e+01,7.700000e+01,2.700000e+01,9.500000e+00,1.175000e+02,0,0.0
1,posts,1.178500e+03,5.400250e+03,4.221750e+03,-5.154125e+03,1.173288e+04,12,12.0
2,followers,5.300000e+07,1.557500e+08,1.027500e+08,-1.011250e+08,3.098750e+08,7,7.0
3,avg_likes,8.775000e+05,4.200000e+06,3.322500e+06,-4.106250e+06,9.183750e+06,2,2.0
4,60_day_eng_rate,1.270000e+00,4.160000e+00,2.890000e+00,-3.065000e+00,8.495000e+00,4,4.0
5,new_post_avg_likes,9.550000e+05,4.600000e+06,3.645000e+06,-4.512500e+06,1.006750e+07,2,2.0
6,total_likes,2.357500e+09,1.225000e+10,9.892500e+09,-1.248125e+10,2.708875e+10,9,9.0


In [ ]:
# handle outlier (capping)

def iqr_capping(df, columns):
    df = df.copy()

    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)

        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        df[col] = df[col].clip(
            lower=lower,
            upper=upper
        )
    
    return df


In [10]:
# split data
x = df.drop(columns=['followers'])
y = df['followers']

from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)